# ComfyUI on Google Colab v5e-1 TPU
Select **Runtime → Change runtime type → TPU (v5e-1)** before running. Cache import/export is local and portable; Google Drive is optional only for models.


In [ ]:
import os, platform, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/kevinmetten/ComfyUI-TPU.git'
# Use 'master' after PR #1 is merged. This value selects the existing PR branch for validation.
BRANCH = 'codex/build-comfyui-backend-for-google-colab-tpu'
# Leave unset for requirements.txt's upstream comfy-kitchen. Set a Git URL only after TPU profiling justifies the fork.
COMFY_KITCHEN_SPEC = None
ROOT = Path('/content/ComfyUI-TPU')
CACHE_DIR = ROOT / '.cache' / 'tpu_xla'
tpu_detected = bool(os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME') or os.path.exists('/dev/accel0'))
assert tpu_detected, 'Select a v5e-1 TPU runtime before continuing'
print('Repository:', REPO_URL)
print('Branch:', BRANCH)
print('Python:', platform.python_version())
print('TPU detection:', tpu_detected)
print('Cache directory:', CACHE_DIR)


## Install the TPU runtime
The official PyTorch/XLA TPU package index resolves a mutually compatible runtime. Re-running skips packages that are already importable.


In [ ]:
import importlib.metadata, importlib.util

def installed_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

def versions_compatible():
    torch_version = installed_version('torch')
    xla_version = installed_version('torch-xla')
    return bool(torch_version and xla_version and torch_version.split('+')[0].split('.')[:2] == xla_version.split('+')[0].split('.')[:2])

if not versions_compatible():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'torch_xla[tpu]', 'torchvision', 'torchaudio', '-f', 'https://storage.googleapis.com/libtpu-releases/index.html'])

assert versions_compatible(), f"Incompatible torch {installed_version('torch')} and torch_xla {installed_version('torch-xla')}; their major/minor versions must match"
print('Torch:', installed_version('torch'))
print('Torch/XLA:', installed_version('torch-xla'))
for package in ('torchvision', 'torchaudio', 'libtpu'):
    print(f'{package}:', installed_version(package) or 'not installed as a Python distribution')


In [ ]:
import re, tempfile
if not (ROOT / '.git').exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, str(ROOT)])
else:
    subprocess.check_call(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', str(ROOT), 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', str(ROOT), 'reset', '--hard', f'origin/{BRANCH}'])
requirements = (ROOT / 'requirements.txt').read_text().splitlines()
requirements = [line for line in requirements if not re.match(r'^\s*(torch|torchvision|torchaudio)(?:\s|[<>=!~]|$)', line, re.I)]
with tempfile.NamedTemporaryFile('w', suffix='.txt', delete=False) as filtered:
    filtered.write('\n'.join(requirements) + '\n')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', filtered.name])
if COMFY_KITCHEN_SPEC:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', COMFY_KITCHEN_SPEC])
os.chdir(ROOT)
print('Comfy-Kitchen:', importlib.metadata.version('comfy-kitchen'))
print('ComfyUI commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## Optional cache import
Run this before any TPU operation. Cancel the upload dialog to start with an empty cache.


In [ ]:
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        subprocess.check_call([sys.executable, '-m', 'tools.tpu_cache', 'import', name])
        break


## Capability probe
Review every operation. An `error` is an unsupported path, not a passed test.


In [ ]:
PROBE_REPORT = ROOT / 'tpu_probe.json'
DIAGNOSTICS = ROOT / 'diagnostics'
subprocess.check_call([sys.executable, '-m', 'tools.tpu_probe', '--output', str(PROBE_REPORT), '--diagnostics', str(DIAGNOSTICS)])


## Optional model storage
Local `/content` paths work directly. Mount Drive only if desired, then configure its directories with `extra_model_paths.yaml`. Hugging Face downloads should be explicitly initiated by the user.


In [ ]:
# Optional:
# from google.colab import drive
# drive.mount('/content/drive')


## Launch and tunnel
This downloads Cloudflare's tunnel binary and starts ComfyUI. The printed `trycloudflare.com` URL is the remote UI.


In [ ]:
import re, time, urllib.request
cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o755)
startup_log = open(ROOT / 'comfyui-tpu-startup.log', 'w')
server = subprocess.Popen([sys.executable, 'main.py', '--tpu', '--listen', '0.0.0.0'], cwd=ROOT, stdout=startup_log, stderr=subprocess.STDOUT)
time.sleep(10)
assert server.poll() is None, f'ComfyUI exited early; inspect {ROOT / 'comfyui-tpu-startup.log'}'
tunnel = subprocess.Popen([str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8188'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    print(line, end='')
    match = re.search(r'https://[^ ]+\.trycloudflare\.com', line)
    if match:
        print('ComfyUI URL:', match.group(0)); break
print('Preserve:', PROBE_REPORT, DIAGNOSTICS, ROOT / 'comfyui-tpu-startup.log')


## Export cache to your computer


In [ ]:
archive = ROOT / 'comfyui-v5e-tpu-cache.zip'
subprocess.check_call([sys.executable, '-m', 'tools.tpu_cache', 'export', str(archive)])
files.download(str(archive))
